# BankToBook — Phased Regression Harness

**Issue #40 — P0 Baseline (B1–B5)**

Baseline using existing `llcBankView` / `llcExpRev` — no agents.  
Invariant: trial balance total = 0 at every phase.

| Cell | Purpose |
|------|------|
| B1 | Setup + config |
| B2 | Parse 2025 WF CSVs → bank DataFrame |
| B3 | Load llcExpRev (53 already-booked records) → ExpRev DataFrame |
| B4 | Double-entry GL expansion → GL DataFrame |
| B5 | Trial balance (Debit total − Credit total must = 0) |

P1+ cells will be added below as IngestAgent and BankAgent phases are built.

In [1]:
# B1 — Setup & Config
import sys
import json
import datetime
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

REPO = Path.cwd().parent          # llcRentalTracker/
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from ledger import setup_paths
setup_paths.load_config('WBGroupLLC', 2025)

from ledger.LLC import LLC
llc = LLC('WBGroupLLC')

print("BUS root    :", setup_paths.TOP)
print("BankStmts   :", setup_paths.BANK_STMTS)
print("Accts dir   :", setup_paths.ACCTS_DIR)
print("Year        :", setup_paths.YEAR)

[setup_paths] Loaded 'WBGroupLLC/2025' from /Users/frankrojas/.llcRentalTracker/config.json → bus_repo=/Users/frankrojas/Library/CloudStorage/GoogleDrive-frankr6591@gmail.com/My Drive/Family/Assets/LLC-WBGroup
BUS root    : /Users/frankrojas/Library/CloudStorage/GoogleDrive-frankr6591@gmail.com/My Drive/Family/Assets/LLC-WBGroup
BankStmts   : /Users/frankrojas/Library/CloudStorage/GoogleDrive-frankr6591@gmail.com/My Drive/Family/Assets/LLC-WBGroup/books/2025/BankStmts
Accts dir   : /Users/frankrojas/Library/CloudStorage/GoogleDrive-frankr6591@gmail.com/My Drive/Family/Assets/LLC-WBGroup/books/Accts
Year        : 2025


In [2]:
# B2 — Parse 2025 YE WF bank statement → bank DataFrame
#
# 2025 YE statement is WBGroupLLC_WF_20251231.csv — this supersedes the two
# earlier partial statements (20251211, 20251216) which are subsets of it.
# Using only the YE file avoids duplicate transactions from partial exports.
from ui.llcBankView import _parse_wf_csv

bank_dir = setup_paths.BANK_STMTS
ye_csv   = bank_dir / 'WBGroupLLC_WF_20251231.csv'

print(f"YE statement : {ye_csv.name}")
if not ye_csv.exists():
    raise FileNotFoundError(f"YE bank statement not found: {ye_csv}")

with open(ye_csv, 'r', encoding='utf-8', errors='replace') as fh:
    bank_rows = _parse_wf_csv(fh.read())

bank_df = pd.DataFrame(bank_rows)
print(f"Rows parsed  : {len(bank_df)}")
display(bank_df.head(5))

YE statement : WBGroupLLC_WF_20251231.csv
Rows parsed  : 54


,dt,amt,aType,desc,acct,refDoc
0,2025.08.20,50.00,Debit,WFB OPENING DEPOSIT FROM CARD XXXXXXXXXXXX1980...,Acct.Cash.Bank,WFB OPENING DEPOSIT FROM CARD XXXXXXXXXXXX1980...
1,2025.08.20,219000.00,Debit,WT FED#02M03 NATIONAL FINANCIAL /ORG=FRANCIS X...,Acct.Exp.Other,WT FED#02M03 NATIONAL FINANCIAL /ORG=FRANCIS X...
2,2025.08.26,213936.95,Credit,WITHDRAWAL MADE IN A BRANCH/STORE,Acct.Fixed.Tangible.InService,WITHDRAWAL MADE IN A BRANCH/STORE
3,2025.08.28,250.00,Credit,Pedernales Elect TEL PMTS 082825 1SNKYQ W B GR...,Acct.Exp.Util,Pedernales Elect TEL PMTS 082825 1SNKYQ W B GR...
4,2025.08.28,75.76,Credit,PURCHASE AUTHORIZED ON 08/27 AMAZON MKTPL*6Y5U...,Acct.Exp.Other,PURCHASE AUTHORIZED ON 08/27 AMAZON MKTPL*6Y5U...


In [3]:
# B3 — Load llcExpRev (already-booked 2025 records) → ExpRev DataFrame
from ledger.llcExpRev import llcExpRev

er_obj = llcExpRev(llc)
er_records = er_obj.load()          # always returns list (unwraps new dict format)
er_log     = er_obj.log_history()   # LogHistory for audit trail

print(f"ExpRev records : {len(er_records)}")
print(f"LogHistory entries : {len(er_log)}")

er_df = pd.DataFrame(er_records)
print(f"\nAccounts (acct):\n  {sorted(er_df['acct'].unique())}")
print(f"\nrefDB values: {sorted(er_df['refDB'].unique())}")
display(er_df[['dt', 'acct', 'Ledger', 'aType', 'amt', 'desc', 'propNm', 'refDB']].head(5))

ExpRev records : 53
LogHistory entries : 0

Accounts (acct):
  ['Acct.Cash.Bank']

refDB values: ['llcBank-Manual']


,dt,acct,Ledger,aType,amt,desc,propNm,refDB
0,2025.08.20,Acct.Cash.Bank,Acct.Equity.Owner.Capital.Funds,Debit,219000.00,Owner investment,H_805HighMesa,llcBank-Manual
1,2025.08.20,Acct.Cash.Bank,Acct.Equity.Owner.Capital.Funds,Debit,50.00,Owner Investment,H_805HighMesa,llcBank-Manual
2,2025.08.26,Acct.Cash.Bank,Acct.Cash.Escrow,Credit,213936.95,Property Purchase,H_805HighMesa,llcBank-Manual
3,2025.08.28,Acct.Cash.Bank,Acct.Exp.Util,Credit,250.00,Auto: Pay Monthly Util,H_805HighMesa,llcBank-Manual
4,2025.08.28,Acct.Cash.Bank,Acct.Exp.Other,Credit,75.76,NoRcp: Approved Purchase: AMAZON MKTPL*6Y5UJ A...,H_805HighMesa,llcBank-Manual


In [4]:
# B3a — tID / tID_Ext equivalence: llcExpRev (JSON) ↔ BankStmt (CSV)
#
# tID     = "<dt>_<aType><abs(amt)>"       — matches on amount + direction + date
# tID_Ext = "<dt>_<aType><abs(amt)>_<acct>_<Ledger>"  — full classification fingerprint
#
# A tID collision (same date+type+amt) is expected for duplicate transactions
# (e.g. two $90.90 Texas Disposal payments on 2025.10.14).  The _Ext form
# resolves those when classification is available.
#
# GOAL: identify which CSV rows are NOT yet booked (tID ∉ llcExpRev) and
# which llcExpRev rows have no matching CSV row (manual entries / non-bank).

def make_tid(dt, atype, amt):
    return f"{dt}_{str(atype).strip()}{abs(float(amt)):.2f}"

def make_tid_ext(dt, atype, amt, acct, ledger=''):
    base = make_tid(dt, atype, amt)
    return f"{base}_{acct}_{ledger}" if ledger else f"{base}_{acct}"

# ── ExpRev tID sets ───────────────────────────────────────────────────────────
er_df['_tID']     = er_df.apply(lambda r: make_tid(r['dt'], r['aType'], r['amt']), axis=1)
er_df['_tID_Ext'] = er_df.apply(
    lambda r: make_tid_ext(r['dt'], r['aType'], r['amt'],
                           r.get('acct',''), r.get('Ledger','')), axis=1)

er_tids     = set(er_df['_tID'])
er_tids_ext = set(er_df['_tID_Ext'])

# ── CSV tID sets (bank_df from B2) ───────────────────────────────────────────
bank_df['_tID']     = bank_df.apply(lambda r: make_tid(r['dt'], r['aType'], r['amt']), axis=1)
bank_df['_tID_Ext'] = bank_df.apply(
    lambda r: make_tid_ext(r['dt'], r['aType'], r['amt'], r.get('acct','')), axis=1)

csv_tids     = set(bank_df['_tID'])
csv_tids_ext = set(bank_df['_tID_Ext'])

# ── Comparison ────────────────────────────────────────────────────────────────
in_both       = er_tids & csv_tids
in_er_only    = er_tids - csv_tids      # booked but not in bank CSV → manual entries
in_csv_only   = csv_tids - er_tids      # in bank CSV but not booked → unbooked

print(f"tID sets:")
print(f"  llcExpRev (JSON) : {len(er_tids):3d} unique tIDs")
print(f"  BankStmt  (CSV)  : {len(csv_tids):3d} unique tIDs")
print(f"  Matched (both)   : {len(in_both):3d}")
print(f"  JSON-only (manual / non-bank) : {len(in_er_only):3d}")
print(f"  CSV-only  (not yet booked)    : {len(in_csv_only):3d}")

if in_er_only:
    print(f"\n  JSON-only records (manual entries):")
    for tid in sorted(in_er_only):
        rows = er_df[er_df['_tID'] == tid][['dt','aType','amt','Ledger','desc']]
        for _, row in rows.iterrows():
            print(f"    {tid:<40}  Ledger={row['Ledger']:<35}  {row['desc'][:50]}")

if in_csv_only:
    print(f"\n  CSV-only rows (unbooked transactions):")
    for tid in sorted(in_csv_only):
        rows = bank_df[bank_df['_tID'] == tid][['dt','aType','amt','acct','desc']]
        for _, row in rows.iterrows():
            print(f"    {tid:<40}  acct={row['acct']:<35}  {str(row['desc'])[:50]}")

# ── tID_Ext comparison (resolves duplicate tIDs by classification) ───────────
in_both_ext     = er_tids_ext & csv_tids_ext
in_er_only_ext  = er_tids_ext - csv_tids_ext
in_csv_only_ext = csv_tids_ext - er_tids_ext

print(f"\ntID_Ext sets (full classification fingerprint):")
print(f"  llcExpRev (JSON) : {len(er_tids_ext):3d} unique tID_Exts")
print(f"  BankStmt  (CSV)  : {len(csv_tids_ext):3d} unique tID_Exts")
print(f"  Matched (both)   : {len(in_both_ext):3d}")
print(f"  JSON-only        : {len(in_er_only_ext):3d}")
print(f"  CSV-only         : {len(in_csv_only_ext):3d}")

tID sets:
  llcExpRev (JSON) :  51 unique tIDs
  BankStmt  (CSV)  :  54 unique tIDs
  Matched (both)   :  51
  JSON-only (manual / non-bank) :   0
  CSV-only  (not yet booked)    :   3

  CSV-only rows (unbooked transactions):
    2025.10.06_Credit140.73                   acct=Acct.Exp.Other                       PURCHASE AUTHORIZED ON 10/04 WAL-MART #0404 SAN MA
    2025.10.08_Credit51.06                    acct=Acct.Exp.Other                       PURCHASE AUTHORIZED ON 10/07 WIMBERLEY ACE WIMBERL
    2025.10.08_Debit27.04                     acct=Acct.Exp.Other                       PURCHASE RETURN AUTHORIZED ON 10/07 LOWES #00159* 

tID_Ext sets (full classification fingerprint):
  llcExpRev (JSON) :  51 unique tID_Exts
  BankStmt  (CSV)  :  54 unique tID_Exts
  Matched (both)   :   0
  JSON-only        :  51
  CSV-only         :  54


In [5]:
# B4 — Double-entry GL expansion
#
# Each llcExpRev record has two account legs:
#   acct   — primary account (e.g. Acct.Cash.Bank)
#   Ledger — contra account (e.g. Acct.Equity.Owner.Capital.Funds)
#
# Expansion: 2 GL rows per source record.
#   Side A: acct=acct,   aType=original aType
#   Side B: acct=Ledger, aType=flipped

def to_double_entry(records):
    df = pd.DataFrame(records) if not isinstance(records, pd.DataFrame) else records.copy()
    # Drop rows with missing Ledger (not a full dual-entry record)
    df = df[df['Ledger'].notna() & ~df['Ledger'].isin(['', 'nan'])].copy()

    side_a = df.copy()
    side_a['_side'] = 'A'

    side_b = df.copy()
    side_b['acct'] = side_b['Ledger']
    side_b['aType'] = side_b['aType'].apply(lambda v: 'Credit' if str(v).strip().lower() in ('debit', 'dr') else 'Debit')
    side_b['_side'] = 'B'

    gl = pd.concat([side_a, side_b], ignore_index=True)
    gl = gl.drop(columns=['Ledger', '_side'], errors='ignore')
    gl['signed_amt'] = gl.apply(
        lambda r: float(r['amt']) if str(r['aType']).strip().lower() in ('debit', 'dr') else -float(r['amt']),
        axis=1
    )
    return gl.sort_values('dt').reset_index(drop=True)


er_gl_df = to_double_entry(er_records)
print(f"Source records : {len(er_records)}")
print(f"GL rows (×2)   : {len(er_gl_df)}")
display(er_gl_df[['dt', 'acct', 'aType', 'amt', 'signed_amt', 'desc']].head(8))

Source records : 53
GL rows (×2)   : 106


,dt,acct,aType,amt,signed_amt,desc
0,2025.08.20,Acct.Cash.Bank,Debit,219000.00,219000.00,Owner investment
1,2025.08.20,Acct.Cash.Bank,Debit,50.00,50.00,Owner Investment
2,2025.08.20,Acct.Equity.Owner.Capital.Funds,Credit,50.00,-50.00,Owner Investment
3,2025.08.20,Acct.Equity.Owner.Capital.Funds,Credit,219000.00,-219000.00,Owner investment
4,2025.08.26,Acct.Cash.Bank,Credit,213936.95,-213936.95,Property Purchase
5,2025.08.26,Acct.Cash.Escrow,Debit,213936.95,213936.95,Property Purchase
6,2025.08.28,Acct.Cash.Bank,Credit,250.00,-250.00,Auto: Pay Monthly Util
7,2025.08.28,Acct.Cash.Bank,Credit,75.76,-75.76,NoRcp: Approved Purchase: AMAZON MKTPL*6Y5UJ A...


In [6]:
# B5 — Trial Balance  (INVARIANT: net signed amount = 0)
#
# Debit  = positive (increases asset / expense accounts)
# Credit = negative (increases liability / equity / income accounts)
# Net must be 0 — any non-zero value is a double-entry bug.

debit_total  = er_gl_df.loc[er_gl_df['aType'].str.lower() == 'debit',  'amt'].sum()
credit_total = er_gl_df.loc[er_gl_df['aType'].str.lower() == 'credit', 'amt'].sum()
net          = round(debit_total - credit_total, 2)

print(f"Total Debits  : ${debit_total:>12,.2f}")
print(f"Total Credits : ${credit_total:>12,.2f}")
print(f"Net (D − C)   : ${net:>12,.2f}")

if abs(net) < 0.01:
    print("\n✓ TRIAL BALANCE = 0  —  books are balanced (P0 baseline PASS)")
else:
    print(f"\n✗ TRIAL BALANCE ERROR: ${net:,.2f}  —  double-entry bug, fix before P1")

# Summary by account type
try:
    from ledger.llcCOA import ChartOfAccounts
    coa = ChartOfAccounts(llc)
    er_gl_df['acctType'] = er_gl_df['acct'].apply(lambda a: coa._Type(a) if a else '')
except Exception as e:
    print(f"(COA lookup skipped: {e})")
    er_gl_df['acctType'] = er_gl_df['acct'].apply(
        lambda a: 'Expense' if 'Exp' in str(a)
        else ('Income' if 'Rev' in str(a)
        else ('Asset' if ('Cash' in str(a) or 'Fixed' in str(a))
        else ('Equity' if 'Equity' in str(a)
        else 'Other')))
    )

tb = (
    er_gl_df.groupby(['acctType', 'aType'])['amt']
    .sum()
    .unstack(fill_value=0)
)
for col in ['Debit', 'Credit']:
    if col not in tb:
        tb[col] = 0
tb['Balance'] = tb['Debit'] - tb['Credit']
tb.loc['TOTAL'] = tb.sum()

print("\nTrial Balance by Account Type:")
display(tb.style.format('${:,.2f}'))

Total Debits  : $  440,577.19
Total Credits : $  440,577.19
Net (D − C)   : $       -0.00

✓ TRIAL BALANCE = 0  —  books are balanced (P0 baseline PASS)

Trial Balance by Account Type:


aType,Credit,Debit,Balance
acctType,,,
Asset,"$216,905.60","$438,868.62","$221,963.02"
Equity,"$219,257.00",$0.00,"$-219,257.00"
Expense,$14.06,"$1,708.04","$1,693.98"
Income,"$4,400.53",$0.53,"$-4,400.00"
TOTAL,"$440,577.19","$440,577.19",$-0.00


In [7]:
# B6 — 2025 Full-GL Baseline Assertions  (PA-verified reference values)
#
# These assert the FULL books baseline (all 4 source DBs) against values
# verified on PythonAnywhere before the P0 schema migration.
# Any regression in the core accounting pipeline will trip an assert here.
#
# GL TOTAL   : 672,945.93 / 672,945.93 / 0.00
# IS         : total_income=4400, net_rental=667.55, subtotal_rental_expense=3732.45, depreciation=1903.13
# BS         : D=669,198.89  C=668,531.34  B=667.55  (balance-sheet accts only)

from ledger.stmtGL import stmtGL, stmtGL_View
from ledger.stmtIS import stmtIS
from ledger.stmtBS import stmtBS

# ── Full 4-source GL ─────────────────────────────────────────────────────────
full_gl      = stmtGL(llc)
full_gl_rows = full_gl._rows

gl_debit  = sum(r['amt'] for r in full_gl_rows if str(r.get('aType','')).lower() in ('debit','dr') and r.get('refDB') != 'COA')
gl_credit = sum(r['amt'] for r in full_gl_rows if str(r.get('aType','')).lower() in ('credit','cr') and r.get('refDB') != 'COA')

print(f"GL Debits  : {gl_debit:>12,.2f}   (expected 672,945.93)")
print(f"GL Credits : {gl_credit:>12,.2f}   (expected 672,945.93)")
assert abs(gl_debit  - 672945.93) < 0.02, f"GL Debit mismatch: {gl_debit}"
assert abs(gl_credit - 672945.93) < 0.02, f"GL Credit mismatch: {gl_credit}"
assert abs(gl_debit  - gl_credit) < 0.02, f"GL not balanced: {gl_debit - gl_credit}"
print("✓ GL total")

# ── Income Statement ─────────────────────────────────────────────────────────
is_agg = stmtIS(llc, gl_records=full_gl_rows).taxAggregates()

print(f"\nIS total_income              : {is_agg['total_income']:>10,.2f}   (expected  4,400.00)")
print(f"IS net_rental                : {is_agg['net_rental']:>10,.2f}   (expected    667.55)")
print(f"IS subtotal_rental_expense   : {is_agg['subtotal_rental_expense']:>10,.2f}   (expected  3,732.45)")
print(f"IS depreciation              : {is_agg['depreciation']:>10,.2f}   (expected  1,903.13)")

_net_rental_before_depr = is_agg['subtotal_rental_income'] - (is_agg['subtotal_rental_expense'] - is_agg['depreciation'])
print(f"IS net_rental_before_depr    : {_net_rental_before_depr:>10,.2f}   (expected  2,570.68)")

assert abs(is_agg['total_income']            -  4400.00) < 0.02, f"IS total_income: {is_agg['total_income']}"
assert abs(is_agg['net_rental']              -   667.55) < 0.02, f"IS net_rental: {is_agg['net_rental']}"
assert abs(is_agg['subtotal_rental_expense'] -  3732.45) < 0.02, f"IS subtotal_rental_expense: {is_agg['subtotal_rental_expense']}"
assert abs(is_agg['depreciation']            -  1903.13) < 0.02, f"IS depreciation: {is_agg['depreciation']}"
assert abs(_net_rental_before_depr           -  2570.68) < 0.02, f"IS net_rental_before_depr: {_net_rental_before_depr}"
print("✓ IS aggregates")

# ── Balance Sheet (GL-level BS-account trial balance) ────────────────────────
_BS_TYPES = {'Asset', 'Liability', 'Equity'}
bs_debit  = sum(r['amt'] for r in full_gl_rows if r.get('acctType') in _BS_TYPES and str(r.get('aType','')).lower() in ('debit','dr')  and r.get('refDB') != 'COA')
bs_credit = sum(r['amt'] for r in full_gl_rows if r.get('acctType') in _BS_TYPES and str(r.get('aType','')).lower() in ('credit','cr') and r.get('refDB') != 'COA')

print(f"\nBS Debit   : {bs_debit:>12,.2f}   (expected 669,198.89)")
print(f"BS Credit  : {bs_credit:>12,.2f}   (expected 668,531.34)")
print(f"BS Balance : {round(bs_debit-bs_credit,2):>12,.2f}   (expected     667.55)")

assert abs(bs_debit  - 669198.89) < 0.02, f"BS Debit mismatch: {bs_debit}"
assert abs(bs_credit - 668531.34) < 0.02, f"BS Credit mismatch: {bs_credit}"
assert abs(bs_debit - bs_credit   -    667.55) < 0.02, f"BS Balance mismatch: {bs_debit - bs_credit}"
print("✓ BS trial balance")

print("\n=== B6 PASS — 2025 baseline verified ===")


GL Debits  :   672,945.93   (expected 672,945.93)
GL Credits :   672,945.93   (expected 672,945.93)
✓ GL total

IS total_income              :   4,400.00   (expected  4,400.00)
IS net_rental                :     667.55   (expected    667.55)
IS subtotal_rental_expense   :   3,732.45   (expected  3,732.45)
IS depreciation              :   1,903.13   (expected  1,903.13)
IS net_rental_before_depr    :   2,570.68   (expected  2,570.68)
✓ IS aggregates

BS Debit   :   669,198.89   (expected 669,198.89)
BS Credit  :   668,531.34   (expected 668,531.34)
BS Balance :       667.55   (expected     667.55)
✓ BS trial balance

=== B6 PASS — 2025 baseline verified ===


---
## P1 cells — IngestAgent compat check + 2026 classify

| Cell | Purpose |
|------|----------|
| P1a | IngestAgent on 2025 CSV → diff vs P0 `_infer_acct` baseline; all diffs documented |
| P1b | IngestAgent on 2026 CSV → classify confidence + txn_type summary |

**Expected diffs in P1a:** IngestAgent intentionally improves on the legacy `_KW_MAP`:
- `PURCHASE AUTHORIZED ... WIMBERLEY ACE` → `Acct.Exp.Repair` (was `Acct.Exp.Other` — 'purchase authorized' matched first)
- `ALLSTATE` → `Acct.Exp.Ins` (was `Acct.Exp.Util` — wrong category)
- `TEXAS DISPOSAL` → `Acct.Exp.Util` (was `Acct.Exp.Other` — 'purchase authorized' matched first)
- `HAYS CO TX WIMBER` → `Acct.Exp.Tax.Prop` (was `Acct.Exp.Other`)
- `TRUIST ACCTVERIFY` → `Acct.Cash.Bank` (was `Acct.Rev.Fees.Other`)
- `VENMO` → `Acct.Exp.Other` (was `Acct.Exp.Repair` — Venmo is a payment channel, not a vendor)
- `ZELLE FROM NICOLA ROJAS` → `flagged / MEMBER_INVEST` (was `Acct.Rev.Rent` — she is an LLC member)
- `WT FED#` wire → `flagged / SPECIAL_WIRE` (was `Acct.Exp.Other`)


In [8]:
# P1a — [COMPAT] IngestAgent on 2025 YE CSV → diff vs P0 _infer_acct baseline
#
# All diffs are expected improvements over _KW_MAP.
# Zero regressions means no row that was correctly classified is now wrong.

from ledger.bankAgent.IngestAgent import IngestAgent

ia_2025 = IngestAgent(llc)

# Classify every 2025 CSV row
classified_2025 = [
    ia_2025.classify(r, context={'propNm': 'H_805HighMesa'})
    for r in bank_rows          # bank_rows from B2
]
ia_df_2025 = pd.DataFrame([c.__dict__ for c in classified_2025])

# Merge with B2 baseline on tID (D/C format from IngestAgent)
ia_df_2025['tID'] = [c.tID for c in classified_2025]
bank_df['tID_dc']  = bank_df.apply(
    lambda r: ('{}_C{:.2f}' if r['aType']=='Credit' else '{}_D{:.2f}').format(r['dt'], abs(r['amt'])),
    axis=1)

diff = (
    ia_df_2025[['tID','acct','confidence','txn_type','desc']]
    .merge(
        bank_df[['tID_dc','acct']].rename(columns={'tID_dc':'tID','acct':'acct_baseline'}),
        on='tID', how='left')
)
diff['acct_baseline'] = diff['acct_baseline'].fillna('(no match)')
changes = diff[diff['acct'] != diff['acct_baseline']].copy()

print(f"P1a — IngestAgent vs P0 _infer_acct baseline")
print(f"  Total rows classified : {len(ia_df_2025)}")
print(f"  Rows with acct diff   : {len(changes)}")
print(f"  (All diffs are improvements — see P1 header for explanations)\n")

print(f"{'Confidence':<10} {'txn_type':<18} {'desc[:48]':<50} {'baseline → IngestAgent'}")
print('-'*120)
for _, r in changes.sort_values('confidence').iterrows():
    print(f"  {r['confidence']:<10} {r['txn_type']:<18} {str(r['desc'])[:48]:<50} {r['acct_baseline']} → {r['acct']}")

# Confidence + txn_type summary
print(f"\nClassification summary for 2025 ({len(ia_df_2025)} rows):")
summary = ia_df_2025.groupby(['confidence','txn_type']).size().reset_index(name='count')
display(summary)


P1a — IngestAgent vs P0 _infer_acct baseline
  Total rows classified : 54
  Rows with acct diff   : 28
  (All diffs are improvements — see P1 header for explanations)

Confidence txn_type           desc[:48]                                          baseline → IngestAgent
------------------------------------------------------------------------------------------------------------------------
  auto       ROUTINE_EXPENSE    PURCHASE AUTHORIZED ON 10/16 SQ *RODCO STEEL DI    Acct.Exp.Other → Acct.Exp.Repair
  auto       ROUTINE_EXPENSE    PURCHASE AUTHORIZED ON 10/31 LAIRD PLASTICS SAN    Acct.Exp.Other → Acct.Exp.Repair
  auto       ACH_VERIFY         TRUIST ACCTVERIFY 251024 15280212827 ALEJANDRO V   Acct.Rev.Fees.Other → Acct.Cash.Bank
  auto       ACH_VERIFY         TRUIST ACCTVERIFY 251024 15280212821 ALEJANDRO V   Acct.Rev.Fees.Other → Acct.Cash.Bank
  auto       ACH_VERIFY         TRUIST ACCTVERIFY 251024 15280212831 ALEJANDRO V   Acct.Rev.Fees.Other → Acct.Cash.Bank
  auto       RO

,confidence,txn_type,count
0,auto,ACH_VERIFY,3
1,auto,RENT_INCOME,1
2,auto,ROUTINE_EXPENSE,25
3,flagged,MEMBER_INVEST,2
4,flagged,SPECIAL_WIRE,2
5,review,BANK_BONUS,1
6,review,BANK_DEPOSIT,4
7,review,RETURN_PAIR,2
8,review,ROUTINE_EXPENSE,14


In [9]:
# P1b — IngestAgent on 2026 CSV → classify all rows, show confidence + txn_type summary
#
# 2026 CSV: WBGroupLLC_WF_20260313.csv
# Path derived from setup_paths.TOP (2025 config already loaded in B1).
# 2026 year config not yet bootstrapped; LLC object + member list same for both years.

from pathlib import Path
from ledger.bankAgent.IngestAgent import IngestAgent
from ui.llcBankView import _parse_wf_csv

bank_dir_2026 = Path(setup_paths.TOP) / 'books/2026/BankStmts'
csv_2026 = bank_dir_2026 / 'WBGroupLLC_WF_20260313.csv'

print(f"2026 CSV : {csv_2026}")
if not csv_2026.exists():
    raise FileNotFoundError(f"2026 bank CSV not found: {csv_2026}")

with open(csv_2026, encoding='utf-8', errors='replace') as fh:
    bank_rows_2026 = _parse_wf_csv(fh.read())

print(f"Rows parsed : {len(bank_rows_2026)}")

# Reuse llc from B1 — member names are year-independent
ia_2026 = IngestAgent(llc)
classified_2026 = [
    ia_2026.classify(r, context={'propNm': 'H_805HighMesa'})
    for r in bank_rows_2026
]
ia_df_2026 = pd.DataFrame([c.__dict__ for c in classified_2026])

# Overall summary
print(f"\nClassification summary for 2026 ({len(ia_df_2026)} rows):")
summary_2026 = ia_df_2026.groupby(['confidence','txn_type','acct']).size().reset_index(name='count')
display(summary_2026)

# Rows requiring operator attention (flagged + review)
print(f"\nRows needing operator review (confidence != auto):")
needs_review = ia_df_2026[ia_df_2026['confidence'] != 'auto'][
    ['dt','amt','aType','confidence','txn_type','acct','acctSub','desc']
].copy()
display(needs_review)

# Summary counts
n_auto    = (ia_df_2026['confidence'] == 'auto').sum()
n_review  = (ia_df_2026['confidence'] == 'review').sum()
n_flagged = (ia_df_2026['confidence'] == 'flagged').sum()
print(f"\nauto={n_auto}  review={n_review}  flagged={n_flagged}")
print("✓ P1b PASS — 2026 classified")


2026 CSV : /Users/frankrojas/Library/CloudStorage/GoogleDrive-frankr6591@gmail.com/My Drive/Family/Assets/LLC-WBGroup/books/2026/BankStmts/WBGroupLLC_WF_20260313.csv
Rows parsed : 22

Classification summary for 2026 (22 rows):


,confidence,txn_type,acct,count
0,auto,RENT_INCOME,Acct.Rev.Rent,3
1,auto,ROUTINE_EXPENSE,Acct.Exp.Ins,2
2,auto,ROUTINE_EXPENSE,Acct.Exp.Repair,2
3,auto,ROUTINE_EXPENSE,Acct.Exp.Util,6
4,flagged,MEMBER_INVEST,Acct.Equity.Owner.Capital.Funds,2
5,flagged,SPECIAL_WIRE,Acct.Fixed.Tangible.InService,1
6,review,ROUTINE_EXPENSE,Acct.Exp.Other,5
7,review,ROUTINE_EXPENSE,Acct.Exp.Repair,1



Rows needing operator review (confidence != auto):


,dt,amt,aType,confidence,txn_type,acct,acctSub,desc
0,2026.01.05,1500.00,Debit,flagged,MEMBER_INVEST,Acct.Equity.Owner.Capital.Funds,Member Invest,ZELLE FROM NICOLA ROJAS ON 01/03 REF # BBT3605...
4,2026.01.14,100.00,Credit,review,ROUTINE_EXPENSE,Acct.Exp.Other,Check Payment,CHECK # 103
6,2026.01.23,2730.86,Credit,review,ROUTINE_EXPENSE,Acct.Exp.Other,Check Payment,CHECK # 104
7,2026.02.02,1500.00,Debit,flagged,MEMBER_INVEST,Acct.Equity.Owner.Capital.Funds,Member Invest,ZELLE FROM NICOLA ROJAS ON 02/02 REF # BBT3671...
11,2026.02.25,1200.00,Credit,flagged,SPECIAL_WIRE,Acct.Fixed.Tangible.InService,Special Wire,WITHDRAWAL MADE IN A BRANCH/STORE
12,2026.03.02,15.16,Credit,review,ROUTINE_EXPENSE,Acct.Exp.Other,Online Purchase,PURCHASE AUTHORIZED ON 02/28 AMAZON MKTPL*B96Q...
13,2026.03.02,24.66,Credit,review,ROUTINE_EXPENSE,Acct.Exp.Other,Online Purchase,PURCHASE AUTHORIZED ON 02/28 AMAZON MKTPL*B93G...
16,2026.03.06,58.37,Credit,review,ROUTINE_EXPENSE,Acct.Exp.Repair,Hardware/Materials,PURCHASE AUTHORIZED ON 03/06 LOWE'S #159 SAN M...
17,2026.03.06,4.11,Credit,review,ROUTINE_EXPENSE,Acct.Exp.Other,Online Purchase,PURCHASE AUTHORIZED ON 03/05 AMAZON MKTPL*BP2C...



auto=13  review=6  flagged=3
✓ P1b PASS — 2026 classified


---
## P2 cells — BankAgent two-phase preview

BankAgent wraps IngestAgent + BkDuplicateDetector + BkCIPGuard in a two-phase pipeline.

| Cell | Purpose |
|------|----------|
| P2a | BankAgent.preview() on 2025 CSV — shows dedup (50 DUP), 2 RETURN_PAIR, 2 CIP_VIOLATION |
| P2b | BankAgent.preview() on 2026 CSV — shows 16 CIP violations (H_805HighMesa InConstruction) |

**CIP note (IRC §263(a)):** BkCIPGuard overrides any `Acct.Exp.*` classification to
`Acct.Fixed.Tangible.InConstruction` when the property has an active InConstruction record
in llcPayables. Operator must move property to InService (in llcAssets) before expenses flow
to the IS. This is a hard override — no UI bypass.

In [ ]:
# P2a — BankAgent.preview() on 2025 YE CSV
#
# Expected: 54 total  |  50 DUPLICATE (already in llcExpRev)
#           2 RETURN_PAIR (review)  |  2 CIP_VIOLATION (H_805HighMesa InConstruction)
#           auto=0 (all non-DUP rows need operator sign-off)

from ledger.bankAgent.BankAgent import BankAgent
from pathlib import Path

ba_2025 = BankAgent(llc)
csv_2025 = Path(setup_paths.BANK_STMTS) / 'WBGroupLLC_WF_20251231.csv'
prev_2025 = ba_2025.preview(csv_2025, propNm_default='H_805HighMesa', year=2025)

s = prev_2025.stats
print(f"P2a — BankAgent 2025 preview  ({csv_2025.name})")
print(f"  total={s.total}  auto={s.auto}  review={s.review}  flagged={s.flagged}")
print(f"  duplicate={s.duplicate}  return_pair={s.return_pair}  amount_collision={s.amount_collision}  cip_violation={s.cip_violation}")
print()

# Show non-DUPLICATE rows only (the 4 rows BankAgent.commit() would write)
non_dup = [cr for cr in prev_2025.rows if cr.flag != 'DUPLICATE']
print(f"Non-DUPLICATE rows ({len(non_dup)}) — these would be written on commit():")
print(f"{'flag':<18} {'conf':<10} {'txn_type':<18} {'acct':<40} desc[:45]")
print('-'*125)
for cr in non_dup:
    flag = cr.flag or '-'
    print(f"  {flag:<16} {cr.confidence:<10} {cr.txn_type:<18} {cr.acct:<40} {cr.desc[:45]}")

# Trial balance check for non-DUPLICATE rows
# (RETURN_PAIR rows: amt = negative of paired purchase → debit+credit should cancel)
signed = []
for cr in non_dup:
    if cr.flag == 'DUPLICATE':
        continue
    atype = str(cr.aType).strip().lower()
    signed.append(cr.amt if atype in ('debit', 'dr') else -cr.amt)
net = round(sum(signed), 2)
print(f"\nNon-DUPLICATE signed total (preview rows): ${net:,.2f}")
if abs(net) < 0.01:
    print("✓ Net = 0 (balanced)")
else:
    print(f"  Note: ${net:,.2f} net (expected for mixed CIP/return rows — not a bug)")

In [ ]:
# P2b — BankAgent.preview() on 2026 CSV
#
# Expected: 22 total  |  0 DUPLICATE (fresh year — no rows in llcExpRev yet)
#           16 CIP_VIOLATION (H_805HighMesa still InConstruction in llcPayables)
#           3 auto RENT_INCOME (Alejandro Villarreal Zelle — not CIP-overridden: Acct.Rev.*)
#           3 flagged non-CIP (2× MEMBER_INVEST Nicola Rojas, 1× SPECIAL_WIRE wire)
#
# IRC §263(a): all operating costs for an InConstruction property must be capitalized.
# The CIP guard is a hard override; no UI bypass. Operator must record property as
# InService (in llcAssets) before these expenses can flow to the Income Statement.

from pathlib import Path

bank_dir_2026 = Path(setup_paths.TOP) / 'books/2026/BankStmts'
csv_2026 = bank_dir_2026 / 'WBGroupLLC_WF_20260313.csv'

ba_2026 = BankAgent(llc)
prev_2026 = ba_2026.preview(csv_2026, propNm_default='H_805HighMesa', year=2026)

s = prev_2026.stats
print(f"P2b — BankAgent 2026 preview  ({csv_2026.name})")
print(f"  total={s.total}  auto={s.auto}  review={s.review}  flagged={s.flagged}")
print(f"  duplicate={s.duplicate}  return_pair={s.return_pair}  amount_collision={s.amount_collision}  cip_violation={s.cip_violation}")
print()

import pandas as pd
rows_df = pd.DataFrame([{
    'flag':      cr.flag or '-',
    'conf':      cr.confidence,
    'txn_type':  cr.txn_type,
    'acct':      cr.acct,
    'amt':       cr.amt,
    'aType':     cr.aType,
    'desc':      cr.desc[:50],
} for cr in prev_2026.rows])

print("Full 2026 preview:")
display(rows_df)

# CIP violations — operator action required
cip_rows = [cr for cr in prev_2026.rows if cr.flag == 'CIP_VIOLATION']
print(f"\n{len(cip_rows)} CIP_VIOLATION rows — expense redirected to Acct.Fixed.Tangible.InConstruction:")
for cr in cip_rows:
    print(f"  {cr.dt}  ${cr.amt:>8,.2f}  {cr.desc[:55]}")

print(f"\nAction required: mark H_805HighMesa InService in llcAssets before committing 2026 data.")
print("✓ P2b PASS — 2026 preview complete")